<a href="https://colab.research.google.com/github/katevaoglou/AI-Monkeys/blob/main/AI-Monkeys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt # Import matplotlib for plotting

# 1. Προετοιμασία Δεδομένων (Input: x, Output: y)
# Συνολικά 3 μεταβλητές εισόδου και μία δυαδική έξοδο (0 ή 1)
# Κάθε γραμμή εισόδου αποτελείται από 3 μεταβλητές [x1, x2, x3]
# x1: Στόμμα: 0=Δόντια, 1=Χαμογελάει, 2=Λυπημένη, 3=Ανοιχτό, 4=Γλώσσα
# x2: Μάτια: 0=Ανοιχτά, 1=Ένα μάτι κλειστό, 2=Μάτια Χ, 3=Κλειστά
# x3: Αξεσουάρ: 0=Όχι, 1=Ναι (Γυαλιά, Μάσκα, Φιόγκος, Τσιρότο)
# x_train: Χαρακτηριστικά για τις 27 μαϊμούδες των δεδομένων εκπαίδευσης
x_train = np.array([
        [0, 0, 0], # 01 - Δαγκώνει
        [0, 1, 0], # 02 - Δαγκώνει
        [0, 3, 0], # 04
        [1, 0, 0], # 05 - Δαγκώνει
        [1, 2, 0], # 07
        [2, 0, 0], # 09 - Δαγκώνει
        [2, 1, 0], # 10 - Δαγκώνει
        [2, 3, 0], # 12
        [3, 1, 0], # 14 - Δαγκώνει
        [3, 2, 0], # 15 - Δαγκώνει
        [3, 3, 0], # 16 - Δαγκώνει
        [4, 0, 0], # 17 - Δαγκώνει
        [4, 2, 0], # 19
        [0, 1, 1], # 22
        [0, 2, 1], # 23
        [0, 3, 0], # 24
        [1, 0, 1], # 25
        [1, 0, 0], # 28 - Δαγκώνει
        [2, 1, 1], # 30
        [2, 3, 1], # 32
        [3, 0, 1], # 33 - Δαγκώνει
        [3, 2, 0], # 35 - Δαγκώνει
        [3, 3, 1], # 36 - Δαγκώνει
        [4, 0, 1], # 37
        [4, 1, 1], # 38
        [4, 2, 1], # 39
        [4, 3, 0]  # 40
    ], dtype=int)

# Δυαδική έξοδος (0=Δεν δαγκώνει, 1=Δαγκώνει)
y_train = np.array([1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0], dtype=int)

# Κωδικοποιήσεις των δεδομένων εισόδου επειδή είναι ονομαστικά κατηγορικά (nominal categorical)
# Π.χ. για το στόμα: 0=Δόντια, 1=Χαμογελάει, 2=Λυπημένη, 3=Ανοιχτό, 4=Γλώσσα
# Το "Χαμογελάει" δεν έχει όμως κάποια σειρά διάταξης σε σχέση με το "Δόντια"
# Ούτε το "Γλώσσα" (4) είναι διπλάσιο από το "Λυπημένη" (2)
# One-hot encode x1 (Στόμα)
x1_train_one_hot = tf.one_hot(x_train[:, 0], depth=5) # x1 έχει 5 κατηγορίες (0-4)
# One-hot encode x2 (Μάτια)
x2_train_one_hot = tf.one_hot(x_train[:, 1], depth=4) # x2 έχει 4 κατηγορίες (0-3)
# Κρατάμε το x3 (Αξεσουάρ) ως την αρχική ακέραια τιμή του
x3_train_original = x_train[:, 2:3]
# Συνδυάζουμε το one-hot x1, το one-hot x2 και το αρχικό x3
x_train = tf.concat([x1_train_one_hot, x2_train_one_hot, tf.cast(x3_train_original, tf.float32)], axis=1)

# 2. Σχεδιασμός του Νευρωνικού Δικτύου
# Επίπεδο εισόδου: Input(shape=(x_train.shape[1],)) για x_train.shape[1] εισόδους (5 για x1, 4 για x2, 1 για x3)
# Ορίζουμε ένα ενδιάμεσο επίπεδο (layer) με 2 νευρώνες (units)
# Επίπεδο εξόδου: 1 νευρώνας με συνάρτηση ενεργοποίησης sigmoid (για δυαδική κατηγοριοποίηση)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(x_train.shape[1],)), # Προτεινόμενος τρόπος για τον ορισμό του input shape, θα είναι (10,)
    tf.keras.layers.Dense(units=2, activation='relu', name='hidden_layer_1'), # Πρώτο ενδιάμεσο επίπεδο
    tf.keras.layers.Dense(units=1, activation='sigmoid', name='output_layer')
])

# 3. Ρύθμιση της Εκπαίδευσης (Optimizer & Loss Function)
model.compile(
    optimizer='sgd', # Ο βελτιστοποιητής: 'sgd' (Stochastic Gradient Descent) είναι ένας απλός και συχνά αποτελεσματικός αλγόριθμος για την προσαρμογή των βαρών του δικτύου.
    loss='binary_crossentropy', # Η συνάρτηση απώλειας: 'binary_crossentropy' χρησιμοποιείται για προβλήματα δυαδικής ταξινόμησης, μετρά πόσο καλά το μοντέλο προβλέπει δυαδικές εξόδους.
    metrics=['accuracy'] # Οι μετρικές αξιολόγησης: 'accuracy' για να παρακολουθούμε την ακρίβεια του μοντέλου κατά την εκπαίδευση.
)

# 4. Εκπαίδευση (Training)
# Το μοντέλο θα προσπαθήσει να μάθει τη συσχέτιση σε 500 επαναλήψεις (epochs)
print("Η εκπαίδευση ξεκινά...")
history = model.fit(x_train, y_train, epochs=500, verbose=0) # Αποθηκεύουμε το ιστορικό εκπαίδευσης
print("Η εκπαίδευση ολοκληρώθηκε!")

# 5. Οπτικοποίηση της απώλειας (Loss Visualization) κατά την εκπαίδευση
plt.plot(history.history['loss'])
plt.title('Συνάρτηση απώλειας του μοντέλου κατά την εκπαίδευση')
plt.ylabel('Απώλεια (Loss)')
plt.xlabel('Εποχή εκπαίδευσης (Epoch)')
plt.show()

In [ ]:
# 6. Έλεγχος του νευρωνικού δικτύου
# Ζητάμε από το δίκτυο να κάνει πρόβλεψη για τα δεδομένα ελέγχου
# Οι μαϊμούδες των δεδομένων ελέγχου είναι οι: 03, 06, 08, 11, 13, 18, 20, 21, 26, 27, 29, 31, 34
# Συνολικά 3 μεταβλητές εισόδου και μία δυαδική έξοδο (0 ή 1)
# Κάθε γραμμή εισόδου αποτελείται από 3 μεταβλητές [x1, x2, x3]
# x1: Στόμμα: 0=Δόντια, 1=Χαμογελάει, 2=Λυπημένη, 3=Ανοιχτό, 4=Γλώσσα
# x2: Μάτια: 0=Ανοιχτά, 1=Ένα μάτι κλειστό, 2=Μάτια Χ, 3=Κλειστά
# x3: Αξεσουάρ: 0=Όχι, 1=Ναι (Γυαλιά, Μάσκα, Φιόγκος, Τσιρότο)
x_test = np.array([
    [0, 2, 0], # 03
    [1, 1, 0], # 06 - Δαγκώνει
    [1, 3, 0], # 08
    [2, 2, 0], # 11
    [3, 0, 0], # 13 - Δαγκώνει
    [4, 1, 0], # 18 - Δαγκώνει
    [4, 3, 0], # 20
    [1, 0, 1], # 21
    [1, 1, 1], # 26
    [1, 2, 0], # 27
    [2, 0, 1], # 29
    [2, 2, 1], # 31
    [3, 1, 1], # 34 - Δαγκώνει
], dtype=int)

# Δυαδική έξοδος (0=Δεν δαγκώνει, 1=Δαγκώνει)
y_test = np.array([0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1], dtype=int)

# Κωδικοποιήσεις των δεδομένων εισόδου επειδή είναι ονομαστικά κατηγορικά (nominal categorical)
# Π.χ. για το στόμα: 0=Δόντια, 1=Χαμογελάει, 2=Λυπημένη, 3=Ανοιχτό, 4=Γλώσσα
# Το "Χαμογελάει" δεν έχει όμως κάποια σειρά διάταξης σε σχέση με το "Δόντια"
# Ούτε το "Γλώσσα" (4) είναι διπλάσιο από το "Λυπημένη" (2)
# One-hot encode x1 (Στόμα) για τα δεδομένα ελέγχου
x1_test_one_hot = tf.one_hot(x_test[:, 0], depth=5)
# One-hot encode x2 (Μάτια) για τα δεδομένα ελέγχου
x2_test_one_hot = tf.one_hot(x_test[:, 1], depth=4) # x2 έχει 4 κατηγορίες (0-3)
# Κρατάμε το x3 (Αξεσουάρ) ως την αρχική ακέραια τιμή του
x3_test_original = x_test[:, 2:3]
# Συνδυάζουμε το one-hot x1, το one-hot x2 και το αρχικό x3
x_test = tf.concat([x1_test_one_hot, x2_test_one_hot, tf.cast(x3_test_original, tf.float32)], axis=1)

# Οι αριθμοί των μαϊμούδων
x_ids = np.array([3, 6, 8, 11, 13, 18, 20, 21, 26, 27, 29, 31, 34], dtype=int)

# Κάνουμε προβλέψεις για τα δεδομένα ελέγχου (x_test)
predictions_proba = model.predict(x_test) # Έξοδος πιθανότητες (0-1)
predictions_binary = (predictions_proba > 0.5).astype(int).flatten() # Μετατροπή σε δυαδική έξοδο (0 ή 1) και ισοπέδωση σε 1D array

print("Αποτελέσματα Προβλέψεων για τα Δεδομένα Ελέγχου:")
print("--------------------------------------------------")
correct_predictions_count = 0
for i in range(len(x_test)):
    # Εδώ εμφανίζουμε τα επεξεργασμένα χαρακτηριστικά εισόδου (x_test),
    # την ζητούμενη έξοδο (πραγματική) και την πρόβλεψη (προβλεπόμενη τιμή)
    actual_label = y_test[i]
    predicted_label = predictions_binary[i]
    status = "Σωστή" if predicted_label == actual_label else "Λάθος"
    if status == "Σωστή":
        correct_predictions_count += 1
    print(f"ID: {x_ids[i]:2d}, Είσοδος (αρχική): {original_x_test[i]}, Πραγματική: {actual_label}, Προβλεπόμενη: {predicted_label} ({status})")

# Υπολογισμός συνολικού ποσοστού σωστών προβλέψεων
total_samples = len(y_test)
accuracy = (correct_predictions_count / total_samples) * 100
print("--------------------------------------------------")
print(f"Συνολικός αριθμός δειγμάτων ελέγχου: {total_samples}")
print(f"Αριθμός σωστών προβλέψεων: {correct_predictions_count}")
print(f"Ποσοστό σωστών προβλέψεων: {accuracy:.2f}%")

### 7. Οπτικοποίηση της Αρχιτεκτονικής του Νευρωνικού Δικτύου (Neural Network Architecture Visualization)

Για να κατανοήσουμε καλύτερα τη δομή του νευρωνικού μας δικτύου, μπορούμε να το οπτικοποιήσουμε. Αυτό θα δείξει τα διάφορα επίπεδα, τον αριθμό των νευρώνων σε κάθε επίπεδο, και πώς συνδέονται μεταξύ τους.

In [ ]:
from tensorflow.keras.utils import plot_model

# Δημιουργία οπτικοποίησης του μοντέλου
# show_shapes=True: εμφανίζει τα σχήματα εισόδου/εξόδου κάθε επιπέδου
# show_layer_names=True: εμφανίζει τα ονόματα των επιπέδων
# to_file='model_architecture.png': αποθηκεύει το διάγραμμα σε ένα αρχείο εικόνας
plot_model(
    model,
    to_file='model_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='LR' # 'TB' για top-to-bottom, 'LR' για left-to-right
)

# Εμφάνιση της εικόνας απευθείας στο Colab
from IPython.display import Image
Image(filename='model_architecture.png')

### 8. Λεπτομερής Ανασκόπηση του Μοντέλου και Εμφάνιση Βαρών (Weights) και Bias

Για να δούμε αναλυτικά το δίκτυό μας, μπορούμε να χρησιμοποιήσουμε την εντολή `model.summary()` η οποία μας δίνει πληροφορίες για κάθε επίπεδο, τον αριθμό των νευρώνων (μέσω του `Output Shape`) και τον αριθμό των παραμέτρων (που περιλαμβάνουν τα βάρη και τα bias).

Επιπλέον, μπορούμε να εκτυπώσουμε τα συγκεκριμένα βάρη και τα bias για κάθε επίπεδο, ώστε να κατανοήσουμε τις τιμές που έχει μάθει το δίκτυο κατά την εκπαίδευση.

In [ ]:
# 9. Εμφάνιση της ανασκόπησης του μοντέλου (summary)
print("Ανασκόπηση του Νευρωνικού Δικτύου:")
model.summary()

In [ ]:
# 10. Εμφάνιση των βαρών (weights) και των bias για κάθε επίπεδο
print("Βάρη (Weights) και Bias ανά Επίπεδο:")
for i, layer in enumerate(model.layers):
    print(f"\nΕπίπεδο {i+1}: {layer.name}")
    weights = layer.get_weights()
    if len(weights) > 0:
        print("  Βάρη (Weights) Shape:", weights[0].shape)
        print("  Βάρη (Weights):\n", weights[0])
        if len(weights) > 1:
            print("  Bias Shape:", weights[1].shape)
            print("  Bias:\n", weights[1])
    else:
        print("  Δεν υπάρχουν εκπαιδεύσιμα βάρη/bias σε αυτό το επίπεδο (π.χ., επίπεδο εισόδου).")